In [ ]:
# Cell 1
import sys
sys.path.append('..')

import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

from src.models.train import load_features, compute_scale_pos_weight, train_best
from src.features.build_features import get_feature_cols
from src.utils import config

# Cell 2
train = load_features()
feature_cols = get_feature_cols(train)
X, y = train[feature_cols], train['TARGET']
spw = compute_scale_pos_weight(y)
cv = StratifiedKFold(config.N_SPLITS, shuffle=True, random_state=config.RANDOM_STATE)

# Cell 3 — Optuna search
def objective(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 300, 900),
        max_depth=trial.suggest_int('max_depth', 3, 6),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
        min_child_weight=trial.suggest_int('min_child_weight', 1, 20),
        gamma=trial.suggest_float('gamma', 0, 5),
        reg_alpha=trial.suggest_float('reg_alpha', 0, 5),
        reg_lambda=trial.suggest_float('reg_lambda', 0, 10),
        scale_pos_weight=spw, eval_metric='auc',
        n_jobs=-1, random_state=config.RANDOM_STATE,
    )
    model = XGBClassifier(**params)
    oof = cross_val_predict(model, X, y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
    return roc_auc_score(y, oof)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40, show_progress_bar=True)

print('BEST CV AUC:', round(study.best_value, 5))
print('BEST PARAMS:', study.best_params)

# Cell 4 — Full optimization pipeline + save best model
model, best_cols, best_auc = train_best(train, feature_cols)
print(f'Final best CV AUC: {best_auc:.5f} using {len(best_cols)} features')